In [ ]:
import yfinance as yf
import pandas as pd
import ta
import warnings
import requests
import datetime
warnings.filterwarnings('ignore')

def fetch_binance_data(ticker, interval='1m', limit=50000):
    """
    Fetches historical high-frequency data from Binance public API.
    (100% free, no API key required)
    """
    # Convert Yahoo ticker (BTC-USD) to Binance ticker (BTCUSDT)
    symbol = ticker.replace('-', '')
    if not symbol.endswith('T'):
        symbol += 'T'
        
    url = "https://api.binance.com/api/v3/klines"
    klines = []
    end_time = int(datetime.datetime.now().timestamp() * 1000)
    
    print(f"Downloading ~{limit} rows of {interval} data from Binance...")
    
    while len(klines) < limit:
        params = {
            'symbol': symbol,
            'interval': interval,
            'limit': min(1000, limit - len(klines)),
            'endTime': end_time
        }
        res = requests.get(url, params=params)
        data = res.json()
        
        if not data or type(data) == dict:
            break
            
        klines = data + klines
        end_time = data[0][0] - 1 # update end time to fetch previous chunk
        
    df = pd.DataFrame(klines, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'close_time', 'qav', 'num_trades', 'taker_base_vol', 'taker_quote_vol', 'ignore'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)
    
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[col] = df[col].astype(float)
        
    return df[['Open', 'High', 'Low', 'Close', 'Volume']]

def fetch_and_engineer_data(ticker, mode='daily'):
    """
    Fetches crypto data and adds technical indicators based on trading mode.
    """
    print(f"Fetching {mode.upper()} data for {ticker}...")
    
    if mode == 'daily':
        df = yf.download(ticker, period='5y', interval='1d')
    elif mode == 'hourly':
        df = yf.download(ticker, period='730d', interval='1h') # Yahoo max for 1h is 730 days
    elif mode == 'minute':
        # We download 50,000 minutes (approx 34 days) to keep training time reasonable (under 10 mins).
        # Downloading 5 years of minutes would crash a standard PC.
        df = fetch_binance_data(ticker, interval='1m', limit=50000)
    else:
        raise ValueError("Invalid mode selected.")
    
    if df.empty:
        raise ValueError(f"No data found for {ticker}")
        
    df.dropna(inplace=True)
    
    # Squeeze to 1D Series
    close = df['Close'].squeeze()
    high = df['High'].squeeze()
    low = df['Low'].squeeze()
    
    # Fetch Macro-Economic Data (Nasdaq and US Dollar Index)
    print("Fetching Macro-Economic Data (Nasdaq & DXY)...")
    if mode in ['daily', 'hourly']:
        period = '5y' if mode == 'daily' else '730d'
        interval = '1d' if mode == 'daily' else '1h'
        
        nasdaq = yf.download('^IXIC', period=period, interval=interval)['Close']
        dxy = yf.download('DX-Y.NYB', period=period, interval=interval)['Close']
        
        df['Nasdaq'] = nasdaq
        df['DXY'] = dxy
        
        # Stock markets close on weekends and nights, so we forward-fill the last known price
        df['Nasdaq'] = df['Nasdaq'].ffill().bfill()
        df['DXY'] = df['DXY'].ffill().bfill()
    else:
        # For minute-by-minute Binance data, we just use a dummy flatline or fetch daily and ffill
        # To keep it simple and avoid massive API alignment issues on 1m, we leave them as 0 for now
        df['Nasdaq'] = 0
        df['DXY'] = 0

    # 1. Moving Averages
    df['SMA_20'] = ta.trend.sma_indicator(close, window=20)
    df['SMA_50'] = ta.trend.sma_indicator(close, window=50)
    df['EMA_20'] = ta.trend.ema_indicator(close, window=20)
    
    # 2. Momentum Indicators
    df['RSI'] = ta.momentum.rsi(close, window=14)
    df['MACD'] = ta.trend.macd(close)
    df['MACD_Signal'] = ta.trend.macd_signal(close)
    
    # 3. Volatility Indicators
    df['BB_High'] = ta.volatility.bollinger_hband(close, window=20)
    df['BB_Low'] = ta.volatility.bollinger_lband(close, window=20)
    df['ATR'] = ta.volatility.average_true_range(high, low, close, window=14)
    
    # Drop rows with NaN values created by indicators
    df.dropna(inplace=True)
    
    # Clean column names (Squeeze multi-index if necessary from Yahoo)
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    
    print(f"Data shape for {ticker} after feature engineering: {df.shape}")
    return df


In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def create_sequences(data, target, seq_length):
    """
    Creates sequences of length `seq_length` for time-series forecasting.
    data: The feature columns.
    target: The target column to predict (e.g., Close price).
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(target[i + seq_length])
    return np.array(X), np.array(y)


def split_and_scale_data(df, seq_length=60, train_split=0.8):
    """
    Splits the data chronologically and applies MinMaxScaling without data leakage.
    Returns scaled 3D sequences ready for an LSTM.
    """
    # Define features to use
    features = ['Close', 'Volume', 'SMA_20', 'SMA_50', 'EMA_20', 'RSI', 'MACD', 'ATR', 'BB_High', 'BB_Low', 'Nasdaq', 'DXY']
    
    # Extract data
    data_values = df[features].values
    target_values = df['Close'].values.reshape(-1, 1) # Close is what we predict
    
    # 1. Chronological Split (NO LEAKAGE)
    train_size = int(len(data_values) * train_split)
    
    train_data = data_values[:train_size]
    test_data = data_values[train_size:]
    
    train_target = target_values[:train_size]
    test_target = target_values[train_size:]
    
    # 2. Fit Scalers ONLY on training data
    feature_scaler = MinMaxScaler(feature_range=(0, 1))
    target_scaler = MinMaxScaler(feature_range=(0, 1))
    
    scaled_train_data = feature_scaler.fit_transform(train_data)
    scaled_test_data = feature_scaler.transform(test_data)
    
    scaled_train_target = target_scaler.fit_transform(train_target)
    scaled_test_target = target_scaler.transform(test_target)
    
    # 3. Create Sequences (3D arrays: [samples, time_steps, features])
    X_train, y_train = create_sequences(scaled_train_data, scaled_train_target, seq_length)
    X_test, y_test = create_sequences(scaled_test_data, scaled_test_target, seq_length)
    
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    
    return X_train, y_train, X_test, y_test, feature_scaler, target_scaler

if __name__ == "__main__":
    from step1_data import fetch_and_engineer_data
    df = fetch_and_engineer_data('BTC-USD')
    X_train, y_train, X_test, y_test, f_scaler, t_scaler = split_and_scale_data(df)


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Conv1D, MaxPooling1D, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

def build_hybrid_model(input_shape):
    """
    Builds a 1D CNN + GRU Hybrid model for high efficiency and accuracy.
    input_shape: (time_steps, num_features)
    """
    model = Sequential([
        Input(shape=input_shape),
        
        # 1D CNN acts as a rapid feature extractor for short-term volatility/spikes
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        
        # GRU captures the long-term temporal trends (faster and more efficient than LSTM)
        GRU(units=50, return_sequences=True),
        Dropout(0.2),
        GRU(units=50, return_sequences=False),
        Dropout(0.2),
        
        Dense(units=25, activation='relu'),
        Dense(units=1) # Predict the target (Close price)
    ])
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def train_and_save_model(model, X_train, y_train, X_test, y_test, ticker):
    """
    Trains the model with EarlyStopping to prevent overfitting, and saves the best model.
    """
    print(f"Training model for {ticker}...")
    
    # Create directory for models
    if not os.path.exists('models'):
        os.makedirs('models')
        
    model_path = f'models/{ticker}_lstm_model.keras'
    
    # Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    checkpoint = ModelCheckpoint(filepath=model_path, monitor='val_loss', save_best_only=True)
    
    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,
        batch_size=32,
        callbacks=[early_stop, checkpoint],
        verbose=1
    )
    
    print(f"Model saved to {model_path}")
    return model, history

if __name__ == "__main__":
    from step1_data import fetch_and_engineer_data
    from step2_preprocess import split_and_scale_data
    
    ticker = 'BTC-USD'
    df = fetch_and_engineer_data(ticker)
    X_train, y_train, X_test, y_test, f_scaler, t_scaler = split_and_scale_data(df)
    
    # Build model using shape [time_steps, features]
    model = build_hybrid_model((X_train.shape[1], X_train.shape[2]))
    model.summary()
    
    # Train
    train_and_save_model(model, X_train, y_train, X_test, y_test, ticker)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

@tf.function
def compute_gradients(model, inputs):
    with tf.GradientTape() as tape:
        tape.watch(inputs)
        predictions = model(inputs)
    # Get the gradient of the prediction with respect to the input features
    return tape.gradient(predictions, inputs)

def integrated_gradients(model, inputs, baseline, steps=50):
    """
    Calculates the exact mathematical feature importance for a Deep Learning model
    using TensorFlow's native Integrated Gradients.
    """
    # 1. Create steps (alphas) between the baseline (all zeros) and the actual data
    alphas = tf.linspace(start=0.0, stop=1.0, num=steps+1)
    
    # Initialize a tensor to store the integrated gradients
    igrads = np.zeros_like(inputs)
    
    # 2. Calculate the difference between the input and the baseline
    delta = inputs - baseline
    
    # 3. For each sample in the dataset, calculate the path integral
    for i in range(len(inputs)):
        sample_input = inputs[i:i+1]
        sample_baseline = baseline[i:i+1]
        sample_delta = delta[i:i+1]
        
        # Generate the interpolated inputs along the path
        interpolated = [sample_baseline + alpha * sample_delta for alpha in alphas]
        interpolated = tf.concat(interpolated, axis=0)
        
        # Calculate gradients for all interpolated steps
        grads = compute_gradients(model, interpolated)
        
        # Approximate the integral using the trapezoidal rule
        grads = (grads[:-1] + grads[1:]) / 2.0
        avg_grads = tf.reduce_mean(grads, axis=0)
        
        # Multiply the average gradient by the delta to get the final Integrated Gradient
        igrads[i] = (sample_delta * avg_grads).numpy()
        
    return igrads

def explain_with_ig(ticker):
    from step1_data import fetch_and_engineer_data
    from step2_preprocess import split_and_scale_data
    
    print(f"Loading data and model for {ticker}...")
    df = fetch_and_engineer_data(ticker)
    X_train, y_train, X_test, y_test, f_scaler, t_scaler = split_and_scale_data(df)
    
    model = load_model(f'models/{ticker}_lstm_model.keras')
    
    # We use a baseline of zeros (representing "no signal")
    baseline = np.zeros_like(X_test)
    inputs = tf.convert_to_tensor(X_test, dtype=tf.float32)
    baseline = tf.convert_to_tensor(baseline, dtype=tf.float32)
    
    print("Calculating Integrated Gradients (Native TensorFlow)...")
    ig_values = integrated_gradients(model, inputs, baseline, steps=50)
    
    print("\nIntegrated Gradients calculation completed successfully!")
    print(f"IG array shape: {ig_values.shape}")
    
    # Let's aggregate the importance to show the top features
    feature_names = ['Close', 'Volume', 'SMA_20', 'SMA_50', 'EMA_20', 'RSI', 'MACD', 'ATR', 'BB_High', 'BB_Low', 'Nasdaq', 'DXY']
    
    # Sum the absolute importance across all test samples and all 60 timesteps
    global_importance = np.sum(np.abs(ig_values), axis=(0, 1))
    
    # Normalize to 100%
    global_importance = (global_importance / np.sum(global_importance)) * 100
    
    print("\n=== OVERALL FEATURE IMPORTANCE ===")
    importance_dict = {feature_names[i]: global_importance[i] for i in range(len(feature_names))}
    sorted_importance = dict(sorted(importance_dict.items(), key=lambda item: item[1], reverse=True))
    
    for feature, importance in sorted_importance.items():
        print(f"{feature.ljust(10)}: {importance:.2f}% impact")

if __name__ == "__main__":
    explain_with_ig('BTC-USD')


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

from step1_data import fetch_and_engineer_data
from step2_preprocess import split_and_scale_data

def run_backtest(ticker, mode='daily', initial_capital=10000.0, trading_fee=0.001):
    print(f"\n==============================================")
    print(f"INITIALIZING BACKTEST ENGINE FOR {ticker}")
    print(f"==============================================\n")
    
    # 1. Fetch Data
    df = fetch_and_engineer_data(ticker, mode=mode)
    X_train, y_train, X_test, y_test, f_scaler, t_scaler = split_and_scale_data(df)
    
    # 2. Load Model
    model_path = f'models/{ticker}_{mode}_model.keras'
    fallback_path = f'models/{ticker}_lstm_model.keras'
    
    if os.path.exists(model_path):
        model = load_model(model_path)
    elif os.path.exists(fallback_path):
        model = load_model(fallback_path)
    else:
        print("X No model found. Please run the universal predictor first to train a model.")
        return
        
    # 3. Get Predictions for the Test Set
    print("\n[>] Running AI predictions on historical test data...")
    predictions = model.predict(X_test, verbose=0)
    
    predicted_prices = t_scaler.inverse_transform(predictions).flatten()
    actual_prices = t_scaler.inverse_transform(y_test).flatten()
    
    # 4. Simulate Trading
    print(f"\n[>] Simulating Trading with ${initial_capital:,.2f} starting capital...")
    balance_usd = initial_capital
    balance_crypto = 0.0
    
    # Trackers for plotting
    portfolio_history = []
    buy_and_hold_history = []
    
    # Buy and Hold metrics (If we just bought on day 1 and did nothing)
    initial_crypto_price = actual_prices[0]
    buy_and_hold_crypto = (initial_capital * (1 - trading_fee)) / initial_crypto_price
    
    trades_executed = 0
    
    # We loop through the test set. For each day 'i', the model predicts 'i+1'.
    # Note: We need the actual price of day 'i' to execute the trade for 'i+1'.
    # Because of how we sequenced the data, actual_prices[i] is the target day.
    # To avoid lookahead bias, if prediction > current_price, we buy.
    
    # Aligning the timeline:
    # sequence is 60 days. The input ends at day T. target is T+1.
    # We need the price at day T to decide if we buy for T+1.
    
    # Reconstruct the "current day" price array (day T)
    features = ['Close', 'Volume', 'SMA_20', 'SMA_50', 'EMA_20', 'RSI', 'MACD', 'ATR', 'BB_High', 'BB_Low', 'Nasdaq', 'DXY']
    close_idx = features.index('Close')
    
    # X_test shape: (samples, 60, 12). We want the 59th timestep of the 'Close' feature, unscaled.
    current_scaled_close = X_test[:, -1, close_idx].reshape(-1, 1)
    current_actual_prices = t_scaler.inverse_transform(current_scaled_close).flatten()
    
    for i in range(len(predicted_prices)):
        current_price = current_actual_prices[i]
        predicted_tomorrow = predicted_prices[i]
        actual_tomorrow = actual_prices[i] # What actually happened the next day
        
        # Strategy: If predicted price is higher than current price, go ALL IN.
        # If predicted price is lower, SELL EVERYTHING.
        
        if predicted_tomorrow > current_price:
            # BUY SIGNAL
            if balance_usd > 0:
                # We have USD, let's buy crypto
                crypto_bought = (balance_usd * (1 - trading_fee)) / current_price
                balance_crypto += crypto_bought
                balance_usd = 0.0
                trades_executed += 1
        else:
            # SELL SIGNAL
            if balance_crypto > 0:
                # We have Crypto, let's sell for USD
                usd_gained = (balance_crypto * current_price) * (1 - trading_fee)
                balance_usd += usd_gained
                balance_crypto = 0.0
                trades_executed += 1
                
        # Calculate daily net worth
        current_net_worth = balance_usd + (balance_crypto * actual_tomorrow)
        portfolio_history.append(current_net_worth)
        
        # Calculate buy and hold net worth
        bh_net_worth = buy_and_hold_crypto * actual_tomorrow
        buy_and_hold_history.append(bh_net_worth)
        
    final_net_worth = portfolio_history[-1]
    bh_final_net_worth = buy_and_hold_history[-1]
    
    ai_profit_pct = ((final_net_worth - initial_capital) / initial_capital) * 100
    bh_profit_pct = ((bh_final_net_worth - initial_capital) / initial_capital) * 100
    
    print("\n==============================================")
    print(f"BACKTEST RESULTS ({len(predicted_prices)} intervals)")
    print(f"----------------------------------------------")
    print(f"Initial Capital   : ${initial_capital:,.2f}")
    print(f"Total Trades Made : {trades_executed}")
    print(f"----------------------------------------------")
    print(f"AI Strategy Final : ${final_net_worth:,.2f} ({ai_profit_pct:+.2f}%)")
    print(f"Buy & Hold Final  : ${bh_final_net_worth:,.2f} ({bh_profit_pct:+.2f}%)")
    print(f"==============================================\n")
    
    # 5. Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(portfolio_history, label='AI Trading Strategy', color='blue', linewidth=2)
    plt.plot(buy_and_hold_history, label='Buy & Hold Strategy', color='orange', linestyle='--')
    plt.title(f'{ticker} Backtest: AI vs Buy & Hold (Starting ${initial_capital:,.0f})')
    plt.xlabel('Time (Intervals)')
    plt.ylabel('Portfolio Value (USD)')
    plt.legend()
    plt.grid(True)
    
    plot_path = f'models/{ticker}_backtest.png'
    plt.savefig(plot_path)
    print(f"Backtest chart saved to {plot_path}")

if __name__ == "__main__":
    ticker = input("\nEnter ticker to backtest (e.g., BTC-USD): ").upper()
    print("Select Trading Mode:")
    print("1. Daily")
    print("2. Hourly")
    choice = input("Enter 1 or 2: ")
    mode = 'hourly' if choice == '2' else 'daily'
    
    run_backtest(ticker, mode=mode)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

from step1_data import fetch_and_engineer_data
from step2_preprocess import split_and_scale_data
from step3_model import build_hybrid_model, train_and_save_model

def predict_any_crypto(ticker, mode):
    print(f"\n==============================================")
    print(f"INITIALIZING {mode.upper()} PIPELINE FOR {ticker}")
    print(f"==============================================\n")
    
    # 1. Fetch Data
    try:
        df = fetch_and_engineer_data(ticker, mode=mode)
    except Exception as e:
        print(f"[X] Could not fetch data for {ticker}. Error: {e}")
        return
        
    if len(df) < 100:
        print(f"[X] Not enough historical data for {ticker}. Need more data to train.")
        return

    # 2. Process and Split Data
    X_train, y_train, X_test, y_test, f_scaler, t_scaler = split_and_scale_data(df)
    
    # 3. Check if model exists, if not, train it
    model_path = f'models/{ticker}_{mode}_model.keras'
    if os.path.exists(model_path):
        print(f"\n[OK] Pre-trained {mode} model found for {ticker}! Loading it...")
        model = load_model(model_path)
    else:
        print(f"\n[!] No model found. Training a fresh Hybrid CNN+GRU {mode} model for {ticker}...")
        model = build_hybrid_model((X_train.shape[1], X_train.shape[2]))
        model, _ = train_and_save_model(model, X_train, y_train, X_test, y_test, f"{ticker}_{mode}")
        
    # 4. Make a Live Prediction
    if mode == 'daily': timeframe = 'TOMORROW'
    elif mode == 'hourly': timeframe = 'NEXT HOUR'
    else: timeframe = 'NEXT MINUTE'
        
    print(f"\n[>] Predicting the {timeframe} closing price for {ticker}...")
    
    # We need the most recent 60 intervals of data
    features = ['Close', 'Volume', 'SMA_20', 'SMA_50', 'EMA_20', 'RSI', 'MACD', 'ATR', 'BB_High', 'BB_Low', 'Nasdaq', 'DXY']
    recent_data = df[features].tail(60).values
    
    # Scale it using the feature scaler
    scaled_recent_data = f_scaler.transform(recent_data)
    
    # Reshape for the model [samples, time_steps, features]
    live_input = np.array([scaled_recent_data])
    
    # Predict
    scaled_prediction = model.predict(live_input, verbose=0)
    
    # Inverse transform to USD
    future_price = t_scaler.inverse_transform(scaled_prediction)[0][0]
    
    # Current Price
    current_price = float(np.squeeze(df['Close'].values)[-1])
    
    print(f"\n==============================================")
    print(f"{ticker} Live {mode.capitalize()} Report:")
    print(f"----------------------------------------------")
    print(f"Current Price (Now): ${current_price:,.2f}")
    print(f"Predicted Price ({timeframe}): ${future_price:,.2f}")
    
    difference = future_price - current_price
    percent_change = (difference / current_price) * 100
    
    if difference > 0:
        print(f"Forecast: UP by ${difference:,.2f} (+{percent_change:.2f}%)")
    else:
        print(f"Forecast: DOWN by ${abs(difference):,.2f} ({percent_change:.2f}%)")
    print(f"==============================================\n")

if __name__ == "__main__":
    target_coin = input("\nEnter a crypto ticker (e.g., SOL-USD, ETH-USD): ").upper()
    print("\nSelect Trading Mode:")
    print("1. Daily (Swing Trading - Predicts next day)")
    print("2. Hourly (Day Trading - Predicts next hour)")
    print("3. Minute (Scalping - Predicts next minute using Binance)")
    choice = input("Enter 1, 2, or 3: ")
    
    mode_map = {'1': 'daily', '2': 'hourly', '3': 'minute'}
    selected_mode = mode_map.get(choice, 'daily')
    
    predict_any_crypto(target_coin, selected_mode)
